# Phase 1: environment, baselines, and nominal IPPO
This notebook is a command dashboard. The implementation lives in `robosoccer/`; runs are copied to Drive only after metadata says they are complete. Pymunk evaluation is sim-to-sim transfer, not physical deployment.

## 1–2. Initialization variables

In [5]:
from pathlib import Path
import json, os, shutil, sys
REPO_URL = "https://github.com/djdhillxn/football"
REPO_DIR = Path("/content/robot-soccer-transfer")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive" / "RobotSoccerTransfer"
print(REPO_DIR, DRIVE_PROJECT)

/content/robot-soccer-transfer /content/drive/MyDrive/RobotSoccerTransfer


## 3. Runtime and GPU check

In [2]:
!python --version
!nvidia-smi || true
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

Python 3.12.13
Tue Jul 21 05:52:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   43C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------------

## 4. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount(str(DRIVE_MOUNT))
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


## 5. Clone or fast-forward the repository

In [6]:
if REPO_DIR.exists():
    dirty = !git -C {REPO_DIR} status --porcelain
    if dirty:
        raise RuntimeError("Repository has local changes; commit or remove them before pulling.")
    !git -C {REPO_DIR} fetch origin --quiet
    !git -C {REPO_DIR} pull --ff-only
else:
    if REPO_URL.startswith("REPLACE_"):
        raise RuntimeError("Set REPO_URL before cloning.")
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git rev-parse HEAD

Cloning into '/content/robot-soccer-transfer'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 44 (delta 1), reused 41 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 70.39 KiB | 23.46 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/robot-soccer-transfer
3937e9e0ea658e440525f3aa93b2cfda2fa578c1


In [8]:
%pwd

'/content/robot-soccer-transfer'

In [7]:
%ls

configs/    pyproject.toml  reports/          robosoccer/  tests/
notebooks/  README.md       requirements.txt  scripts/     webots/


## 6. Install and verify the package

In [9]:
!python -m pip install -e '.[dev]' -q
from importlib.metadata import version
for package in ["robosoccer-transfer", "torch", "gymnasium", "pettingzoo", "pymunk"]:
    print(package, version(package))
sys.path.insert(0, str(REPO_DIR))
import robosoccer
print("robosoccer", robosoccer.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 951.1/951.1 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.6/317.6 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 147.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 138.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 116.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.5/805.5 kB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/

## 7. Shared artifact helpers

In [10]:
def require_shell_success(label):
    status = int(get_ipython().user_ns.get("_exit_code", 0))
    if status != 0:
        raise RuntimeError(f"{label} failed with shell status {status}")

def latest_run(name):
    pointer = REPO_DIR / "runs" / f"latest_{name}.txt"
    if not pointer.is_file():
        raise FileNotFoundError(f"Missing latest-run pointer: {pointer}")
    return Path(pointer.read_text().strip())

def display_json(path):
    data = json.loads(Path(path).read_text())
    print(json.dumps(data, indent=2)[:5000])
    return data

def copy_completed_run(run_dir):
    run_dir = Path(run_dir)
    metadata = json.loads((run_dir / "run_metadata.json").read_text())
    if metadata.get("status") != "complete":
        raise RuntimeError(f"Refusing to sync incomplete run: {run_dir}")
    destination = DRIVE_PROJECT / "runs" / run_dir.name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(run_dir, destination, dirs_exist_ok=True)
    return destination

def restore_run(name):
    source = DRIVE_PROJECT / "runs" / name
    destination = REPO_DIR / "runs" / name
    if not source.is_dir():
        raise FileNotFoundError(source)
    shutil.copytree(source, destination, dirs_exist_ok=True)
    metadata = json.loads((destination / "run_metadata.json").read_text())
    pointer = REPO_DIR / "runs" / f"latest_{metadata['experiment_name']}.txt"
    pointer.parent.mkdir(parents=True, exist_ok=True)
    pointer.write_text(str(destination.resolve()) + "\n")
    return destination

def sync_reports_and_runs():
    shutil.copytree(REPO_DIR / "reports", DRIVE_PROJECT / "reports", dirs_exist_ok=True)
    for pointer in (REPO_DIR / "runs").glob("latest_*.txt"):
        shutil.copy2(pointer, DRIVE_PROJECT / pointer.name)

## 8. Run the full test suite

In [11]:
!python -m pytest -q
require_shell_success("pytest")

..............................                                           [100%]
30 passed in 11.41s


## 9. Run the CPU smoke configuration

In [12]:
!python -m scripts.train --config configs/smoke_test.yaml
require_shell_success("smoke training")
display_json(latest_run("smoke_test") / "run_metadata.json")

INFO: Training MAPPO with 4 environments for 1,024 steps.
MAPPO:   0% 0/4 [00:00<?, ?it/s]INFO: Update 1/4 | success 0.50 | return 2.92 | FPS 195
MAPPO:  25% 1/4 [00:01<00:03,  1.31s/it, kl=0.000, reward=2.92, success=0.50]INFO: Update 2/4 | success 0.14 | return -2.85 | FPS 228
MAPPO:  50% 2/4 [00:02<00:02,  1.50s/it, kl=0.000, reward=-2.85, success=0.14]INFO: Updated failure-directed sampling over 3 profiles.
INFO: Update 3/4 | success 0.00 | return -5.18 | FPS 209
MAPPO:  75% 3/4 [00:03<00:01,  1.15s/it, kl=0.000, reward=-5.18, success=0.00]INFO: Update 4/4 | success 0.14 | return -2.86 | FPS 219
MAPPO: 100% 4/4 [00:05<00:00,  1.24s/it, kl=0.000, reward=-2.86, success=0.14]INFO: Updated failure-directed sampling over 3 profiles.
MAPPO: 100% 4/4 [00:05<00:00,  1.26s/it, kl=0.000, reward=-2.86, success=0.14]
INFO: Completed run: /content/robot-soccer-transfer/runs/20260721_055932_smoke_test_mappo_seed0
{
  "algorithm": "mappo",
  "command": "/content/robot-soccer-transfer/scripts/trai

{'algorithm': 'mappo',
 'command': '/content/robot-soccer-transfer/scripts/train.py --config configs/smoke_test.yaml',
 'cuda_available': True,
 'evaluation_seeds': {'abstract_test': 30000,
  'curriculum': 20000,
  'transfer_test': 40000,
  'validation': 10000,
  'video': 50000},
 'experiment_name': 'smoke_test',
 'failure_exception': None,
 'git_commit': '3937e9e0ea658e440525f3aa93b2cfda2fa578c1',
 'gpu_name': 'NVIDIA L4',
 'output_artifact_paths': {'best_actor': '/content/robot-soccer-transfer/runs/20260721_055932_smoke_test_mappo_seed0/models/best_actor.ts',
  'best_checkpoint': '/content/robot-soccer-transfer/runs/20260721_055932_smoke_test_mappo_seed0/models/best_checkpoint.pt',
  'environment_steps': 1024,
  'final_actor': '/content/robot-soccer-transfer/runs/20260721_055932_smoke_test_mappo_seed0/models/final_actor.ts',
  'final_checkpoint': '/content/robot-soccer-transfer/runs/20260721_055932_smoke_test_mappo_seed0/models/final_checkpoint.pt',
  'final_validation': {'mean_retur

## 10. Run the PettingZoo API validation only

In [13]:
!python -m pytest -q tests/test_core.py -k parallel_api
require_shell_success("PettingZoo parallel API tests")

..                                                                       [100%]
2 passed, 28 deselected in 2.74s


## 11. Record a random-policy environment video

In [14]:
!python -m scripts.record_video --config configs/base.yaml --baseline random --simulator abstract --episodes 1
require_shell_success("random-policy video")

INFO: Recorded /content/robot-soccer-transfer/runs/manual_baseline_videos/videos/random_abstract_seed50000_out_of_bounds.mp4


## 12–13. Evaluate and summarize all scripted baselines

In [15]:
!python -m scripts.evaluate_baselines --config configs/base.yaml --episodes 100
require_shell_success("baseline evaluation")
baseline_run = latest_run("base")
display_json(baseline_run / "eval" / "baseline_summary.json")

Baselines: 100% 599/600 [01:46<00:00,  2.94it/s]INFO: Completed baseline evaluation: /content/robot-soccer-transfer/runs/20260721_055953_baselines_heuristic_seed0
Baselines: 100% 600/600 [01:47<00:00,  5.59it/s]
{
  "double_chase__abstract": {
    "collision_rate": 1.0,
    "cvar_10_return": -11.134416124482051,
    "episode_count": 100,
    "invalid_action_rate": 0.3513194444534698,
    "mean_action_switches": 13.81,
    "mean_profile_success_rate": 0.3,
    "mean_return": -1.9556093710361058,
    "mean_return_95_ci": [
      -3.24470687449301,
      -0.5696591178974477
    ],
    "mean_time_to_score": 10.586666666666666,
    "median_time_to_score": 6.800000000000001,
    "minimum_profile_success_rate": 0.3,
    "out_of_bounds_rate": 0.54,
    "pass_completion_rate": 0.0,
    "possession_fraction": 1.336027438070284,
    "redundant_chase_fraction": 0.6921485099210508,
    "standard_deviation_return": 6.998989583668088,
    "success_rate": 0.3,
    "success_rate_95_ci": [
      0.21,
 

{'double_chase__abstract': {'collision_rate': 1.0,
  'cvar_10_return': -11.134416124482051,
  'episode_count': 100,
  'invalid_action_rate': 0.3513194444534698,
  'mean_action_switches': 13.81,
  'mean_profile_success_rate': 0.3,
  'mean_return': -1.9556093710361058,
  'mean_return_95_ci': [-3.24470687449301, -0.5696591178974477],
  'mean_time_to_score': 10.586666666666666,
  'median_time_to_score': 6.800000000000001,
  'minimum_profile_success_rate': 0.3,
  'out_of_bounds_rate': 0.54,
  'pass_completion_rate': 0.0,
  'possession_fraction': 1.336027438070284,
  'redundant_chase_fraction': 0.6921485099210508,
  'standard_deviation_return': 6.998989583668088,
  'success_rate': 0.3,
  'success_rate_95_ci': [0.21, 0.39],
  'worst_decile_return': -11.134416124482051},
 'double_chase__pymunk': {'collision_rate': 0.99,
  'cvar_10_return': -10.029467551718913,
  'episode_count': 100,
  'invalid_action_rate': 0.08083871749675926,
  'mean_action_switches': 8.27,
  'mean_profile_success_rate': 0.

## 14. Train nominal IPPO

In [16]:
!python -m scripts.train --config configs/ippo_nominal.yaml
require_shell_success("nominal IPPO training")

2026-07-21 06:01:45.609322: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 06:01:45.673296: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-07-21 06:01:47,423:jax._src.path:41: etils.epath found. Using etils.epath for file I/O.
INFO: Training IPPO with 32 environments for 2,000,000 steps.
IPPO:   0% 0/245 [00:00<?, ?it/s]INFO: Update 1/245 | success 0.04 | return -4.62 | FPS 395
IPPO:   9% 23/245 [07:49<1:14:39, 20.18s/it, kl=0.001, reward=-3.77, success=0.08]INFO: Update 24/245 | success 0.06 | re

## 15. Evaluate IPPO in the abstract simulator

In [17]:
ippo_run = latest_run("ippo_nominal")
!python -m scripts.evaluate --run-dir {ippo_run} --checkpoint best --simulator abstract --suite standard
require_shell_success("IPPO abstract evaluation")

Standard: 100% 300/300 [00:23<00:00, 12.97it/s]
INFO: Evaluation complete | abstract standard | episodes 300 | success 0.453


## 16. Evaluate the same frozen IPPO actor in Pymunk

In [18]:
ippo_run = latest_run("ippo_nominal")
!python -m scripts.evaluate --run-dir {ippo_run} --checkpoint best --simulator pymunk --suite transfer
require_shell_success("IPPO Pymunk transfer evaluation")

Transfer: 100% 100/100 [00:35<00:00,  2.82it/s]
INFO: Evaluation complete | pymunk transfer | episodes 100 | success 0.730


## 17. Record IPPO videos in both simulators

In [19]:
ippo_run = latest_run("ippo_nominal")
!python -m scripts.record_video --run-dir {ippo_run} --checkpoint best --simulator abstract --episodes 3
require_shell_success("IPPO abstract videos")
!python -m scripts.record_video --run-dir {ippo_run} --checkpoint best --simulator pymunk --episodes 3
require_shell_success("IPPO Pymunk videos")

INFO: Recorded /content/robot-soccer-transfer/runs/20260721_060143_ippo_nominal_ippo_seed0/videos/ippo_nominal_abstract_seed50001_success.mp4
INFO: Recorded /content/robot-soccer-transfer/runs/20260721_060143_ippo_nominal_ippo_seed0/videos/ippo_nominal_abstract_seed50000_out_of_bounds.mp4
INFO: Recorded /content/robot-soccer-transfer/runs/20260721_060143_ippo_nominal_ippo_seed0/videos/ippo_nominal_abstract_seed50002_out_of_bounds.mp4
INFO: Recorded /content/robot-soccer-transfer/runs/20260721_060143_ippo_nominal_ippo_seed0/videos/ippo_nominal_pymunk_seed50000_success.mp4
INFO: Recorded /content/robot-soccer-transfer/runs/20260721_060143_ippo_nominal_ippo_seed0/videos/ippo_nominal_pymunk_seed50001_stationary_ball.mp4
INFO: Recorded /content/robot-soccer-transfer/runs/20260721_060143_ippo_nominal_ippo_seed0/videos/ippo_nominal_pymunk_seed50002_success.mp4


## 18. Sync completed artifacts to Drive

In [20]:
for name in ["smoke_test", "ippo_nominal"]:
    print(copy_completed_run(latest_run(name)))
print(copy_completed_run(baseline_run))
sync_reports_and_runs()

/content/drive/MyDrive/RobotSoccerTransfer/runs/20260721_055932_smoke_test_mappo_seed0
/content/drive/MyDrive/RobotSoccerTransfer/runs/20260721_060143_ippo_nominal_ippo_seed0
/content/drive/MyDrive/RobotSoccerTransfer/runs/20260721_055953_baselines_heuristic_seed0


## 19. Finished
Verify the Drive copies, then disconnect and delete the Colab runtime. Full runs are expensive; do not leave an idle accelerator attached.